In [1]:
# PyTorch core library - tensors, autograd (automatic differentiation), and building blocks for neural nets
import torch
# nn module contains layers (Conv2d, Linear), activations (ReLU), and loss functions (CrossEntropyLoss)
import torch.nn as nn
# optim module contains optimizers (Adam, SGD, etc.) that update model weights during training
import torch.optim as optim

# torchvision gives us ready-made computer vision datasets, models, and image transforms
import torchvision
# CIFAR10: a dataset of 60,000 32x32 color images across 10 classes (plane, car, bird, cat, etc.)
from torchvision.datasets import CIFAR10

In [2]:
#Datasets And DataLoaders
# DataLoader wraps a dataset and serves it up in shuffled batches during training/evaluation
from torch.utils.data import DataLoader
# transforms lets us preprocess every image the same way before it reaches the model
import torchvision.transforms as transforms

# Compose chains several transforms together; each image passes through them in order
transform = transforms.Compose([
    # ToTensor: converts a PIL image (pixel values 0-255) into a PyTorch tensor scaled to [0, 1]
    transforms.ToTensor(),
    # Normalize: rescales each of the 3 color channels using mean=0.5, std=0.5
    # this maps [0, 1] pixel values to roughly [-1, 1], which helps the network train faster/more stably
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))


])

# Load the 50,000-image CIFAR-10 training set. download=True fetches it if it's not already on disk
trainset = CIFAR10(root='./data', train=True, download=True, transform=transform)
# Load the 10,000-image CIFAR-10 test set - reserved ONLY for final evaluation, never for training
testset = CIFAR10(root='./data', train=False, download=True, transform=transform)


In [3]:
# batch_size=64: the model looks at 64 images at once before each weight update (a compromise between speed and stable gradients)
# shuffle=True: randomizes image order every epoch so the model doesn't memorize the data's original order
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
# No shuffling needed for the test set since we never train on it, order doesn't matter
testloader = DataLoader(testset, batch_size=64)

### Build The CNN

In [4]:
# Define our Convolutional Neural Network (CNN) architecture
class CNN(nn.Module):
    def __init__(self):
        # Always call the parent nn.Module's __init__ first - sets up internal bookkeeping
        super(CNN, self).__init__()

        # conv_layers: extracts visual features (edges, textures, shapes) from the raw image
        self.conv_layers = nn.Sequential(
            # Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
            # in_channels=3 because CIFAR images are RGB (3 color channels)
            # out_channels=32 means this layer learns 32 different filters (feature detectors)
            # padding=1 keeps the image size the same after convolution (32x32 stays 32x32)
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            # ReLU: activation function, turns negative values to 0, adds non-linearity so the
            # network can learn complex patterns instead of just straight lines
            nn.ReLU(),
            # MaxPool2d(2, 2): halves the spatial size (32x32 -> 16x16), keeping the strongest signal
            nn.MaxPool2d(2, 2),

            # Second conv block: 32 -> 64 feature maps; deeper layers learn more complex/abstract patterns
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Third conv block: 64 -> 128 feature maps
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)


        )


        # fc_layers: fully-connected layers that turn extracted features into class predictions
        self.fc_layers = nn.Sequential(
            # Input size = 4*4*128: after 3 rounds of MaxPool2d(2,2), a 32x32 image becomes 4x4,
            # and at that point we have 128 channels (from the last conv layer)
            nn.Linear(4*4*128, 256),
            nn.ReLU(),
            # Final layer: outputs 10 raw scores (logits), one per CIFAR-10 class
            nn.Linear(256, 10)
        )

    def forward(self, x):
        # Step 1: run the image through the convolutional feature extractor
        x = self.conv_layers(x)
        # Step 2: flatten from shape (batch, 128, 4, 4) to (batch, 128*4*4) so it fits the linear layers
        x = x.view(x.size(0), -1) #flattening
        # Step 3: run the flattened features through the classifier to get 10 class scores
        x = self.fc_layers(x)

        return x
    

In [5]:
# Create an instance of our CNN. This is the "baseline" model - trained on ALL training data,
# with no validation split, so we have something to compare the validation-based run against.
model = CNN()

In [6]:
# CrossEntropyLoss combines softmax + negative log-likelihood into one function.
# It's the standard loss for multi-class classification problems like this one.
criterion = nn.CrossEntropyLoss()

In [7]:
# Adam optimizer: adaptively adjusts the learning rate for each parameter.
# A strong general-purpose default choice for training neural networks.
optimizer = optim.Adam(model.parameters())

### Training The CNN (Baseline - trained on all data, no validation tracking)

In [8]:
# baseline_train_loss stores the average training loss for every epoch, so we can plot/compare it later
baseline_train_loss = []
epochs = 10  # one "epoch" = one full pass over the entire training dataset

for epoch in range(epochs):
    epoch_training_loss = 0.0  # accumulates the total loss across all batches in this epoch

    for images, labels in trainloader:  # DataLoader hands us one batch (64 images + true labels) at a time
        optimizer.zero_grad() #reset gradients - PyTorch accumulates gradients by default, so we clear them each batch

        output = model.forward(images) #FP - forward pass: model produces its predictions (logits) for this batch

        loss = criterion(output, labels) #loss fnx - compares predictions vs true labels, returns one loss number

        loss.backward() #BP - backpropagation: computes the gradient of the loss w.r.t. every model weight

        optimizer.step() # update all model weights using those gradients (one gradient-descent step)

        epoch_training_loss += loss.item() # .item() pulls the loss out of the tensor as a plain Python float
    

    avg_epoch_loss = epoch_training_loss/len(trainloader) # average loss per batch this epoch
    baseline_train_loss.append(avg_epoch_loss) # save it so we can plot it after training finishes
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {avg_epoch_loss}")
        




Epoch 1/10, Training Loss: 1.3629761042497348
Epoch 2/10, Training Loss: 0.9298043433205246
Epoch 3/10, Training Loss: 0.7505855682637076
Epoch 4/10, Training Loss: 0.6149527431677675
Epoch 5/10, Training Loss: 0.5092559502938824
Epoch 6/10, Training Loss: 0.4165953525039546
Epoch 7/10, Training Loss: 0.327363859747758
Epoch 8/10, Training Loss: 0.2535052719190145
Epoch 9/10, Training Loss: 0.18971472733732683
Epoch 10/10, Training Loss: 0.1550774311699221


### Evaluate Our Model (baseline, on the untouched test set)


In [9]:
correct_labels = 0 # running count of correct predictions
total_labels = 0   # running count of total predictions made

model.eval() # switch to evaluation mode - disables training-only behavior (dropout/batchnorm, if any)

with torch.no_grad(): # disables gradient tracking - we're not training here, so this saves memory & time
    for images, labels in testloader:
        outputs = model.forward(images) # get predicted class scores for this batch

        _, predicted = torch.max(outputs, 1) # pick the class with the highest score as the prediction

        correct_labels += (predicted == labels).sum().item() # count matches between prediction and truth
        total_labels += labels.size(0) # add this batch's size to the running total

print(f"Test Accuracy: {100 * correct_labels / total_labels}%")
print(f"Accuracy: {correct_labels}/{total_labels} * 100 = {100 * correct_labels / total_labels}%")



Test Accuracy: 75.19%
Accuracy: 7519/10000 * 100 = 75.19%


### Train / Validation Split

In [10]:
# random_split lets us carve a dataset into two non-overlapping random subsets
from torch.utils.data import random_split

val_ratio = 0.1  # reserve 10% of the training data purely for validation (never trained on)
val_size = int(len(trainset) * val_ratio)   # e.g. 5000 images
train_size = len(trainset) - val_size       # remaining ~45000 images actually used for training

# manual_seed(42) makes this split reproducible - re-running this cell gives the same split every time
train_subset, val_subset = random_split(
    trainset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Separate loaders: shuffle the training subset each epoch, no need to shuffle validation data
train_subset_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64)

print(f"Train: {len(train_subset)}, Val: {len(val_subset)}")

Train: 45000, Val: 5000


### Train CNN With Validation (fresh model, for a fair comparison)

In [ ]:
# Create a brand-new model so this run is independent and fairly comparable to the baseline above
model_val = CNN()
criterion_val = nn.CrossEntropyLoss()
optimizer_val = optim.Adam(model_val.parameters())

epochs = 10
# history stores per-epoch metrics (train loss, val loss, val accuracy) so we can plot progress later
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    model_val.train()  # explicitly set training mode (matters for layers like dropout/batchnorm)
    running_train_loss = 0.0

    # ---- Training phase: learn from the 90% training split only ----
    for images, labels in train_subset_loader:
        optimizer_val.zero_grad()             # clear gradients from the previous batch
        output = model_val.forward(images)    # forward pass: get predictions
        loss = criterion_val(output, labels)  # compute how wrong the predictions are
        loss.backward()                       # backprop: compute gradients
        optimizer_val.step()                  # update weights using those gradients
        running_train_loss += loss.item()     # accumulate this batch's loss

    avg_train_loss = running_train_loss / len(train_subset_loader)  # average loss per batch

    # ---- Validation phase: check performance on the held-out 10%, WITHOUT updating any weights ----
    model_val.eval()          # evaluation mode - turns off training-only behavior
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # no gradients needed - we're only measuring performance, not learning
        for images, labels in val_loader:
            output = model_val.forward(images)       # forward pass on validation images
            loss = criterion_val(output, labels)      # measure how wrong these predictions are
            running_val_loss += loss.item()

            _, predicted = torch.max(output, 1)       # predicted class = index of the highest score
            correct += (predicted == labels).sum().item()  # count correct predictions
            total += labels.size(0)                    # count total predictions made

    avg_val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    # Save this epoch's numbers so we can plot the full training history afterward
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_acc"].append(val_accuracy)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, "
          f"Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

### Plot: Training Loss - With Validation vs Without Validation

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
# Baseline model's training loss - trained on ALL 50000 images, no validation held out
plt.plot(range(1, len(baseline_train_loss)+1), baseline_train_loss,
         label="Without Validation (train loss)", marker='o')
# Second model's training loss - trained on only 45000 images (10% held out for validation)
plt.plot(range(1, len(history["train_loss"])+1), history["train_loss"],
         label="With Validation (train loss)", marker='o')
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss: With vs Without Validation")
plt.legend()
plt.grid(True)
plt.show()

### Plot: Validation Loss & Accuracy

In [ ]:
n_epochs_val = len(history["val_loss"])

# Two y-axes sharing one x-axis: loss (red, left) and accuracy % (blue, right)
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(range(1, n_epochs_val+1), history["val_loss"], color="tab:red", marker='o', label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation Loss", color="tab:red")
ax1.tick_params(axis='y', labelcolor="tab:red")

ax2 = ax1.twinx()  # twinx() creates a second y-axis that shares the same x-axis
ax2.plot(range(1, n_epochs_val+1), history["val_acc"], color="tab:blue", marker='s', label="Val Accuracy")
ax2.set_ylabel("Validation Accuracy (%)", color="tab:blue")
ax2.tick_params(axis='y', labelcolor="tab:blue")

plt.title("Validation Loss & Accuracy per Epoch")
fig.tight_layout()
plt.show()

### Final Evaluation: Test Set Comparison (Without Validation vs With Validation)

In [ ]:
def evaluate(model_to_eval, loader):
    """Runs a model over every batch in a DataLoader and returns accuracy as a percentage."""
    model_to_eval.eval()  # evaluation mode - disables training-only layer behavior
    correct = 0
    total = 0
    with torch.no_grad():  # no gradients needed - we're only measuring, not training
        for images, labels in loader:
            outputs = model_to_eval.forward(images)      # get predicted scores for this batch
            _, predicted = torch.max(outputs, 1)          # pick the highest-scoring class
            correct += (predicted == labels).sum().item() # count correct predictions
            total += labels.size(0)                        # count total predictions
    return 100 * correct / total

# Evaluate both trained models on the SAME untouched test set - this is the fair, final comparison
acc_no_val = evaluate(model, testloader)        # baseline model, trained on all 50000 images
acc_with_val = evaluate(model_val, testloader)  # model trained with a validation split (45000 images)

print(f"Test Accuracy (No Validation Model): {acc_no_val:.2f}%")
print(f"Test Accuracy (With Validation Model): {acc_with_val:.2f}%")

# Bar chart summarizing the final head-to-head comparison
plt.figure(figsize=(5, 5))
plt.bar(["Without Validation", "With Validation"], [acc_no_val, acc_with_val], color=["tab:orange", "tab:green"])
plt.ylabel("Test Accuracy (%)")
plt.title("Final Test Accuracy Comparison")
for i, v in enumerate([acc_no_val, acc_with_val]):
    plt.text(i, v + 0.5, f"{v:.2f}%", ha='center')
plt.ylim(0, 100)
plt.show()